In [8]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
from PIL import Image
import torch
import requests
device = ("cuda"if torch.cuda.is_available() else "cpu")
from transformers import CLIPProcessor, CLIPModel

model = CLIPModel.from_pretrained("geolocal/StreetCLIP").to(device)
processor = CLIPProcessor.from_pretrained("geolocal/StreetCLIP")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: geolocal/StreetCLIP
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

In [12]:
train_dir ='/kaggle/input/competitions/iaio-2026-sf-r-image-classification/train_img/train'
class_names = sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])
num_classes = len(class_names)
class_to_idx = {cls_name: i for i, cls_name in enumerate(class_names)}
idx_to_class = {v: k for k, v in class_to_idx.items()}
class_names

['Australia',
 'Brazil',
 'Canada',
 'France',
 'Japan',
 'Russia',
 'United Kingdom']

In [13]:
test_dir = "/kaggle/input/competitions/iaio-2026-sf-r-image-classification/test_img/test/"
preds = []
for idx, file in enumerate(sorted(os.listdir(test_dir))):
    img = Image.open(os.path.join(test_dir, file)).convert("RGB")
    inputs = processor(text =class_names, images = img, return_tensors = "pt",padding = True).to(device)
    outputs = model(**inputs)
    logits_per_image = outputs.logits_per_image
    probs = logits_per_image.softmax(dim=1)
    pred= probs.argmax(dim= 1)
    preds.append(pred.cpu().numpy())
preds =np.array(preds).flatten().tolist()

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/iaio-2026-sf-r-image-classification/test_img/test'

In [18]:
sub = pd.DataFrame({'path':sorted(os.listdir('/kaggle/input/competitions/iaio-2026-sf-r-image-classification/test_img/test')),'label':preds})
sub['label'] = sub['label'].apply(lambda x: idx_to_class[x])

In [20]:
sub.to_csv("sub1.csv",index= False)